# Batch Document Extraction with Llama Vision (Clean Version)

Streamlined batch processing notebook using modular components.

**Features:**
- Early model loading
- Configurable output directory
- Comprehensive analytics and visualizations
- Clean, modular code structure

## 1. Setup and Configuration

In [ ]:
# Core imports
import os
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display
from rich import print as rprint
from rich.console import Console
from rich.table import Table

warnings.filterwarnings('ignore')
console = Console()

# Import batch processing modules
from common.batch_analytics import BatchAnalytics
from common.batch_processor import BatchDocumentProcessor
from common.batch_reporting import BatchReporter
from common.batch_visualizations import BatchVisualizer
from common.evaluation_metrics import load_ground_truth
from common.extraction_parser import discover_images
from common.ground_truth_evaluator import GroundTruthEvaluator

rprint("[bold green]✅ Modules imported successfully[/bold green]")

In [ ]:
# Configuration
CONFIG = {
    # Model settings
    'MODEL_PATH': "/home/jovyan/nfs_share/models/Llama-3.2-11B-Vision-Instruct",
    
    # Batch settings
    'DATA_DIR': 'evaluation_data',
    'GROUND_TRUTH': 'evaluation_data/ground_truth.csv',
    'MAX_IMAGES': None,  # None for all, or set limit
    'DOCUMENT_TYPES': None,  # None for all, or ['invoice', 'receipt']
    
    # Output settings
    'OUTPUT_BASE': os.getenv('OUTPUT_DIR', 'output'),
    
    # V100 optimization
    'USE_QUANTIZATION': True,
    'DEVICE_MAP': 'auto',
    'MAX_NEW_TOKENS': 4000,
    'TORCH_DTYPE': 'bfloat16',
    'LOW_CPU_MEM_USAGE': True
}

# Prompt configuration
PROMPT_CONFIG = {
    'detection_file': 'prompts/document_type_detection.yaml',
    'detection_key': 'detection',
    'extraction_files': {
        'INVOICE': 'prompts/invoice_extraction.yaml',
        'RECEIPT': 'prompts/receipt_extraction.yaml',
        'BANK_STATEMENT': 'prompts/bank_statement_extraction.yaml'
    },
    'extraction_keys': {
        'INVOICE': 'standard',
        'RECEIPT': 'standard',
        'BANK_STATEMENT': 'flat'
    }
}

rprint("[bold blue]📋 Configuration loaded[/bold blue]")

## 2. Output Directory Setup

In [ ]:
# Setup output directories
OUTPUT_BASE = Path(CONFIG['OUTPUT_BASE'])
if not OUTPUT_BASE.is_absolute():
    OUTPUT_BASE = Path.cwd() / OUTPUT_BASE

BATCH_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

OUTPUT_DIRS = {
    'base': OUTPUT_BASE,
    'batch': OUTPUT_BASE / 'batch_results',
    'csv': OUTPUT_BASE / 'csv',
    'visualizations': OUTPUT_BASE / 'visualizations',
    'reports': OUTPUT_BASE / 'reports'
}

for dir_path in OUTPUT_DIRS.values():
    dir_path.mkdir(parents=True, exist_ok=True)

rprint(f"[cyan]📁 Output base: {OUTPUT_BASE}[/cyan]")
rprint(f"[cyan]⏰ Timestamp: {BATCH_TIMESTAMP}[/cyan]")

## 3. Model Loading

In [ ]:
# Load model once for entire batch
from common.model_loader import load_v100_model
from common.llama_vision_table_extractor import LlamaVisionTableExtractor

rprint("[bold green]🔧 Loading model...[/bold green]")

model, processor = load_v100_model(
    model_path=CONFIG['MODEL_PATH'],
    use_quantization=CONFIG['USE_QUANTIZATION'],
    device_map=CONFIG['DEVICE_MAP'],
    max_new_tokens=CONFIG['MAX_NEW_TOKENS'],
    torch_dtype=CONFIG['TORCH_DTYPE'],
    low_cpu_mem_usage=CONFIG['LOW_CPU_MEM_USAGE']
)

extractor = LlamaVisionTableExtractor(processor=processor, model=model)
rprint("[bold green]✅ Model ready for batch processing![/bold green]")

## 4. Image Discovery

In [ ]:
# Discover and filter images
all_images = discover_images(CONFIG['DATA_DIR'])
ground_truth = load_ground_truth(CONFIG['GROUND_TRUTH'])

rprint(f"[cyan]📷 Found {len(all_images)} images[/cyan]")
rprint(f"[cyan]📊 Loaded {len(ground_truth)} ground truth entries[/cyan]")

# Apply filters
if CONFIG['DOCUMENT_TYPES']:
    filtered = []
    for img in all_images:
        img_name = Path(img).name
        if img_name in ground_truth:
            doc_type = ground_truth[img_name].get('DOCUMENT_TYPE', '').lower()
            if any(dt.lower() in doc_type for dt in CONFIG['DOCUMENT_TYPES']):
                filtered.append(img)
    all_images = filtered
    rprint(f"[yellow]🔍 Filtered to {len(all_images)} images[/yellow]")

if CONFIG['MAX_IMAGES']:
    all_images = all_images[:CONFIG['MAX_IMAGES']]
    rprint(f"[yellow]📸 Limited to {len(all_images)} images[/yellow]")

rprint(f"\n[bold green]Ready to process {len(all_images)} images[/bold green]")

## 5. Batch Processing

In [ ]:
# Initialize processor and evaluator
evaluator = GroundTruthEvaluator(CONFIG['GROUND_TRUTH'])
processor = BatchDocumentProcessor(
    extractor=extractor,
    evaluator=evaluator,
    prompt_config=PROMPT_CONFIG,
    console=console
)

# Process batch
batch_results, processing_times, document_types_found = processor.process_batch(
    all_images, verbose=True, progress_interval=5
)

# Summary
rprint(f"\n[bold green]✅ Processed {len(batch_results)} images[/bold green]")
rprint(f"[cyan]Average time: {np.mean(processing_times):.2f}s[/cyan]")
rprint(f"[cyan]Document types: {document_types_found}[/cyan]")

## 6. Generate Analytics

In [ ]:
# Create analytics
analytics = BatchAnalytics(batch_results, processing_times)

# Generate and save DataFrames
saved_files, df_results, df_summary, df_doctype_stats, df_field_stats = analytics.save_all_dataframes(
    OUTPUT_DIRS['csv'], BATCH_TIMESTAMP, verbose=True
)

# Display summaries
rprint("\n[bold blue]📊 Results Summary[/bold blue]")
display(df_summary)

if not df_doctype_stats.empty:
    rprint("\n[bold blue]📈 Document Type Statistics[/bold blue]")
    display(df_doctype_stats)

if df_field_stats is not None and len(df_field_stats) > 0:
    rprint("\n[bold blue]📊 Top Field Accuracies[/bold blue]")
    display(df_field_stats.head(10))

## 7. Create Visualizations

In [ ]:
# Create visualizations
visualizer = BatchVisualizer()

viz_files = visualizer.create_all_visualizations(
    df_results, 
    df_doctype_stats,
    OUTPUT_DIRS['visualizations'],
    BATCH_TIMESTAMP,
    show=True  # Set to False for headless execution
)

rprint(f"[green]✅ Created {len(viz_files)} visualizations[/green]")

## 8. Generate Reports

In [ ]:
# Generate reports
reporter = BatchReporter(
    batch_results, 
    processing_times,
    document_types_found,
    BATCH_TIMESTAMP
)

# Save all reports
report_files = reporter.save_all_reports(
    OUTPUT_DIRS,
    df_results,
    df_summary,
    df_doctype_stats,
    CONFIG['MODEL_PATH'],
    {
        'data_dir': CONFIG['DATA_DIR'],
        'ground_truth': CONFIG['GROUND_TRUTH'],
        'max_images': CONFIG['MAX_IMAGES'],
        'document_types': CONFIG['DOCUMENT_TYPES']
    },
    {
        'use_quantization': CONFIG['USE_QUANTIZATION'],
        'device_map': CONFIG['DEVICE_MAP'],
        'max_new_tokens': CONFIG['MAX_NEW_TOKENS'],
        'torch_dtype': CONFIG['TORCH_DTYPE'],
        'low_cpu_mem_usage': CONFIG['LOW_CPU_MEM_USAGE']
    },
    verbose=True
)

# Display executive summary
markdown_report = reporter.generate_executive_summary(
    df_results, df_doctype_stats, OUTPUT_DIRS['base']
)
display(Markdown(markdown_report))

## 9. Final Summary

In [ ]:
# Display final summary
console.rule("[bold green]Batch Processing Complete[/bold green]")

summary_table = Table(title="Final Summary")
summary_table.add_column("Metric", style="cyan")
summary_table.add_column("Value", style="green")

total_images = len(batch_results)
successful = len([r for r in batch_results if 'error' not in r])
avg_accuracy = df_results['overall_accuracy'].mean() if len(df_results) > 0 else 0
throughput = 60 / np.mean(processing_times) if processing_times else 0

summary_table.add_row("Total Images", str(total_images))
summary_table.add_row("Success Rate", f"{(successful/total_images*100):.1f}%")
summary_table.add_row("Average Accuracy", f"{avg_accuracy:.2f}%")
summary_table.add_row("Throughput", f"{throughput:.1f} img/min")
summary_table.add_row("Output Directory", str(OUTPUT_BASE))

console.print(summary_table)

rprint(f"\n[bold green]🎉 All results saved to: {OUTPUT_BASE}[/bold green]")
rprint(f"[cyan]📊 CSV files: {len(saved_files)} files[/cyan]")
rprint(f"[cyan]📈 Visualizations: {len(viz_files)} charts[/cyan]")
rprint(f"[cyan]📄 Reports: {len(report_files)} documents[/cyan]")